# BB84 Quantum Key Distribution (PennyLane)

Alice encodes bits in random bases, Bob measures in random bases.
Matching bases form the sifted key; QBER detects eavesdropping.

In [ ]:
import random
import pennylane as qml

NUM_BITS = 16
dev = qml.device("default.qubit", wires=1, shots=1)

## BB84 circuit

In [ ]:
@qml.qnode(dev)
def bb84_circuit(bit, encode_basis, measure_basis):
    if bit == 1:
        qml.PauliX(wires=0)
    if encode_basis == "+":
        qml.Hadamard(wires=0)
    if measure_basis == "+":
        qml.Hadamard(wires=0)
    return qml.sample(wires=0)

print(qml.draw(bb84_circuit)(1, "+", "z"))

## Run BB84

In [ ]:
def run_bb84(eavesdrop=False):
    alice_bits = [random.randint(0, 1) for _ in range(NUM_BITS)]
    alice_bases = [random.choice(["z", "+"]) for _ in range(NUM_BITS)]
    bob_bases = [random.choice(["z", "+"]) for _ in range(NUM_BITS)]

    key_bob = []
    for i in range(NUM_BITS):
        if eavesdrop and random.random() < 0.5:
            _ = bb84_circuit(alice_bits[i], alice_bases[i], random.choice(["z", "+"]))
        result = int(bb84_circuit(alice_bits[i], alice_bases[i], bob_bases[i]))
        key_bob.append(result)

    sift_a = [alice_bits[i] for i in range(NUM_BITS) if alice_bases[i] == bob_bases[i]]
    sift_b = [key_bob[i] for i in range(NUM_BITS) if alice_bases[i] == bob_bases[i]]

    print(f"Sifted key A: {sift_a}")
    print(f"Sifted key B: {sift_b}")
    check = random.sample(range(len(sift_a)), min(4, len(sift_a)))
    errors = sum(1 for i in check if sift_a[i] != sift_b[i])
    qber = errors / len(check) if check else 0
    print(f"QBER: {qber:.2%}")
    if qber > 0.11:
        print("EAVESDROPPING DETECTED!")
    else:
        final = [sift_a[i] for i in range(len(sift_a)) if i not in check]
        print(f"Final key: {final}")

print("=== No eavesdropper ===")
run_bb84(eavesdrop=False)
print("\n=== With eavesdropper ===")
run_bb84(eavesdrop=True)